## CPUC data request
### Bias correct model output with respect to observations with local timezone correction 

This notebook mirrors the localization_methodology notebook, but includes:
- timezone correction to local PST from UTC
- some of the figures are removed
- batch processing **at the end** to run all 70+ stations as requested by CPUC --> IF RUNNING BATCH PROCESS STEP, SKIP TO LAST 2 CELLS

Note: first half of the notebook is validation, second half is to streamline multiple iterations

In [ ]:
import climakitae as ck
import xarray as xr
import pandas as pd
import hvplot as hv

from climakitae.util.utils import read_csv_file, get_closest_gridcell, convert_to_local_time, read_ae_colormap

from xclim.core.calendar import convert_calendar
from xclim.core.units import convert_units_to
from xclim.sdba.adjustment import QuantileDeltaMapping
from xclim.sdba import Grouper

from bokeh.models import HoverTool
from timezonefinder import TimezoneFinder

### Step 1: Read in the station data
Open the HadISD dataset for Sacramento Executive Airport (KSAC) temperatures and do some processing

In [ ]:
# Import stations names and coordinates file
stations = "/home/jovyan/cae-notebooks/collaborative/DFU/wecc-station-data.csv"
stations_df = read_csv_file(stations)
my_station = 'KSLC'
station_id = str(stations_df[stations_df['icao'] == my_station]['station id'].values[0]).replace('-', '')

filepaths = [
    "s3://cadcat/hadisd/HadISD_{}.zarr".format(s_id)
    for s_id in [station_id]
]

obs_ds = xr.open_mfdataset(
    filepaths,
    engine="zarr",
    consolidated=False,
    parallel=True,
    backend_kwargs=dict(storage_options={"anon": True}),
)

obs_ds = obs_ds.tas
obs_ds = convert_units_to(obs_ds, "degF")
obs_ds = obs_ds.chunk(dict(
    time=-1)).compute()

# extract coordinates
lat0 = obs_ds.latitude.values
lon0 = obs_ds.longitude.values
print(lat0, lon0)

In [ ]:
# station timezone conversion
# cannot use the convert_to_local_time function as we are not pulling bias-corrected model data for a station
# but modifying the existing raw station data itself

# brute force replace of 'correct' time

# want 1981 - 2014 as our baseline period
obs_ds_data = obs_ds.loc[(obs_ds.time.dt.year >= 1981) & (obs_ds.time.dt.year <= 2014)]
obs_ds_data.time # to keep

# need to retrieve 2015 to grab the last ~8 hours of "2014" from 2015 (in UTC) to do timezone conversion
obs_ds_tzslice = obs_ds.loc[obs_ds.time.dt.year == 2015]

# combine and convert
obs_ds_total = xr.concat([obs_ds_data, obs_ds_tzslice], dim='time') # 1981-2015

tf = TimezoneFinder()
local_tz = tf.timezone_at(lng=lon0, lat=lat0)
new_time = (pd.DatetimeIndex(obs_ds_total.time)
            .tz_localize("UTC")
            .tz_convert(local_tz)
            .tz_localize(None)
            .astype("datetime64[ns]")
           )
obs_ds_total['time'] = new_time

# subset by initial time
start = obs_ds_data.time[0]
end = obs_ds_data.time[-1]

obs_ds_local = obs_ds_total.sel(time=slice(start, end))

### Step 2: Read in the model output
Here we specifically pick CESM2 because it is known to have a strong warm bias.

In [ ]:
selections = ck.Select()
selections.scenario_historical=['Historical Climate']
selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
selections.append_historical = True
selections.area_average = 'No'
selections.time_slice = (1981, 2060)
selections.resolution = '9 km' # can only use 9km, not 3km for outside of CA regions
selections.timescale = 'hourly'
selections.variable = 'Air Temperature at 2m'
selections.units = 'degF'
selections.area_subset = 'lat/lon'
selections.cached_area = ['coordinate selection']
selections.latitude= (lat0-.03, lat0+.03) # for 9km spacing
selections.longitude= (lon0-.05, lon0+.05) # for 9km spacing
wrf_ds = selections.retrieve()

Timezone conversion for the wrf data

In [ ]:
# convert to local timezone for WRF data
wrf_ds = convert_to_local_time(wrf_ds, selections)

In [ ]:
# moved to after timezone correction
# convert calendar

# need to unchunk for bias correction
wrf_ds = wrf_ds.chunk(dict(time=-1)).compute()
# do some renaming for plotting ease later
wrf_ds.attrs['physical_variable'] = wrf_ds.name
wrf_ds.name = 'Raw'

# Select just CESM2 simulation 
# cesm_sim_name = [sim for sim in wrf_ds.simulation.values if "cesm2" in sim.lower()]
# wrf_ds = wrf_ds.sel(simulation = cesm_sim_name).squeeze()

In [ ]:
# dropping duplicate time indexes and resetting calendar
wrf_ds = wrf_ds.drop_duplicates(dim='time', keep='first')
obs_ds = obs_ds_local.drop_duplicates(dim='time', keep='first')

wrf_ds = convert_calendar(wrf_ds, "noleap")
obs_ds = convert_calendar(obs_ds, "noleap")

# data now processed to go into localization process

### Step 3: Inspect the model data and observations
Here we show record-mean daily-mean temperatures for the observations and raw WRF data to get a sense of the bias in the WRF model.

In [ ]:
def group_ds(ds, obs_ds=obs_ds, projected_ceil=2060):
    
    proj_floor = str(projected_ceil-29)
    proj_ceil = str(projected_ceil)
    
    hist_ds = ds.sel(time=slice(str(obs_ds.time.values[0]),
            str(obs_ds.time.values[-1]))).groupby(
            'time.dayofyear').mean()    
    ssp_ds = ds.sel(time=slice(proj_floor,proj_ceil)).groupby(
            'time.dayofyear').mean()    
    obs_ds = obs_ds.groupby(
        'time.dayofyear').mean()
    
    hist_ds = hist_ds.rename(dict(dayofyear = 'Day of Year'))
    ssp_ds = ssp_ds.rename(dict(dayofyear = 'Day of Year'))
    obs_ds = obs_ds.rename(dict(dayofyear = 'Day of Year'))
    
    return hist_ds, ssp_ds, obs_ds

def compare_raw_and_obs(ds, obs_ds=obs_ds, ylim=(None,None), 
                        width=700, height=300): 
    
    hist_gp, ssp_gp, obs_gp = group_ds(ds, obs_ds)
    
    hist_pl = hist_gp.hvplot(label='Historical raw',c='royalblue')    
    ssp_pl = ssp_gp.hvplot(label='Projected raw',c='goldenrod')    
    obs_pl = obs_gp.hvplot(label='Observations',c='k')
   
    pl = obs_pl * hist_pl * ssp_pl
    pl.opts(ylim=ylim, width=width, height=height, legend_position='right',
           toolbar='below', ylabel=obs_ds.units,title='Record-mean Daily-mean '
            +ds.attrs['physical_variable'])
    
    return pl

compare_raw_and_obs(wrf_ds, obs_ds=obs_ds)

# for multi-model ignore

### Step 4: Perform the bias correction procedure
The next cells define and perform the two operations needed for bias correction:
1. Create the training set, which finds the adjustment factors between the observations and raw model historical data. Note that the raw model output used for training needs to be from the same time period of the observations.
2. Apply these adjustment factors to the raw model historical and projected data.

All training and adjustment is performed on rolling 90-day windows centered on each day of the year (i.e., +/- 45 days) to allow for seasonal adjustment.

In [ ]:
window = 90
def do_QDM(obs, ds, nquantiles=20, 
           group='time.dayofyear', window=window, 
           kind="+"):
    
    group = Grouper(group, window=window)

    ds.attrs['variable'] = ds.name
    ds.name = 'Raw' 
    
    QDM = QuantileDeltaMapping.train(
        obs, 
        ds.sel(
            time=slice(str(obs_ds.time.values[0]),
                       str(obs_ds.time.values[-1]))), 
        nquantiles=nquantiles, 
        group=group, 
        kind=kind)
    
    ds_adj = QDM.adjust(ds).compute()
    
    QDM_ds = QDM.ds.rename(dict(
        dayofyear = 'Day of Year', 
        quantiles='Quantile'))    
    
    ds_adj.name = 'Adjusted' 
    ds_adj = xr.merge([ds, ds_adj])
    
    return QDM_ds,ds_adj


In [ ]:
%%time
adj_factors, adj_ds = do_QDM(obs_ds,wrf_ds)

In [ ]:
adj_ds

## Step 5: Visualize the bias correction results
### Step 5a. Inspect the raw historical WRF quantiles and the adjustment factor for each quantile

In [ ]:
raw_cmap = read_ae_colormap(cmap='ae_orange', cmap_hex=True)
af_cmap = read_ae_colormap(cmap='ae_diverging', cmap_hex=True)

hover_temp = HoverTool(description='Custom Tooltip', 
        tooltips=[('Quantile', '@Quantile'), 
        ('Day of Year', '@{Day_of_Year}'),
                 ('Temperature (degF)', '@hist_q')])
hover_adj = HoverTool(description='Custom Tooltip', 
        tooltips=[('Quantile', '@Quantile'), 
        ('Day of Year', '@{Day_of_Year}'),
                 ('Adjustment (degF)', '@af')])

raw_temp_qs = adj_factors['hist_q'].hvplot.quadmesh(
    x='Quantile',y='Day of Year',z='hist_q').opts(
    tools=[hover_temp], width=425, height=300,
    cmap=raw_cmap, clabel="degF", 
    title = "Temperature quantiles by day of year")
adjf_temp = adj_factors['af'].hvplot.quadmesh(
    x='Quantile',y='Day of Year',z='af').opts(
    tools=[hover_adj], width=425, height=300,
    cmap=af_cmap, clabel="degF",
    title="Adjustment factors for each quantile")

raw_and_af = raw_temp_qs + adjf_temp
raw_and_af.opts(title="Raw historical quantiles"
                + " and computed adjustment factors",
               toolbar='below')

In [ ]:
hv.save(raw_and_af, 'localization_adjfactors_{}.html'.format(my_station), resources="INLINE")

In [ ]:
# drop raw data variable (and unnecessary coordinates?), and rename adjusted back to air temperature
adj_ds = adj_ds['Adjusted']
adj_ds.name = 'Adjusted Air Temperature at 2m'
adj_ds = adj_ds.reset_coords(names=['Lambert_Conformal','x', 'y', 'lakemask', 'landmask', 'lat', 'lon'], drop='True')

# export data
filename = 'localized_airtemp_{}.nc'.format(my_station)
ck.export(adj_ds, filename, 'NetCDF')

---
## BATCH PROCESSING BELOW

In [ ]:
import climakitae as ck
import xarray as xr
import pandas as pd

from climakitae.util.utils import read_csv_file, get_closest_gridcell, convert_to_local_time, read_ae_colormap

from xclim.core.calendar import convert_calendar
from xclim.core.units import convert_units_to
from xclim.sdba.adjustment import QuantileDeltaMapping
from xclim.sdba import Grouper

from bokeh.models import HoverTool
from timezonefinder import TimezoneFinder
import hvplot as hv

# only needs to be read in once
window = 90
def do_QDM(obs, ds, nquantiles=20, 
           group='time.dayofyear', window=window, 
           kind="+"):
    
    group = Grouper(group, window=window)

    ds.attrs['variable'] = ds.name
    ds.name = 'Raw' 
    
    QDM = QuantileDeltaMapping.train(
        obs, 
        ds.sel(
            time=slice(str(obs_ds.time.values[0]),
                       str(obs_ds.time.values[-1]))), 
        nquantiles=nquantiles, 
        group=group, 
        kind=kind)
    
    ds_adj = QDM.adjust(ds).compute()
    
    QDM_ds = QDM.ds.rename(dict(
        dayofyear = 'Day of Year', 
        quantiles='Quantile'))    
    
    ds_adj.name = 'Adjusted' 
    ds_adj = xr.merge([ds, ds_adj])
    
    return QDM_ds,ds_adj

def figure_to_export(adj_factors, stn):
    raw_cmap = read_ae_colormap(cmap='ae_orange', cmap_hex=True)
    af_cmap = read_ae_colormap(cmap='ae_diverging', cmap_hex=True)

    hover_temp = HoverTool(description='Custom Tooltip', 
            tooltips=[('Quantile', '@Quantile'), 
            ('Day of Year', '@{Day_of_Year}'),
                     ('Temperature (degF)', '@hist_q')])
    hover_adj = HoverTool(description='Custom Tooltip', 
            tooltips=[('Quantile', '@Quantile'), 
            ('Day of Year', '@{Day_of_Year}'),
                     ('Adjustment (degF)', '@af')])

    raw_temp_qs = adj_factors['hist_q'].hvplot.quadmesh(
        x='Quantile',y='Day of Year',z='hist_q').opts(
        tools=[hover_temp], width=425, height=300,
        cmap=raw_cmap, clabel="degF", 
        title = "Temperature quantiles by day of year")
    adjf_temp = adj_factors['af'].hvplot.quadmesh(
        x='Quantile',y='Day of Year',z='af').opts(
        tools=[hover_adj], width=425, height=300,
        cmap=af_cmap, clabel="degF",
        title="Adjustment factors for each quantile")

    raw_and_af = raw_temp_qs + adjf_temp
    raw_and_af.opts(title="Raw historical quantiles"
                    + " and computed adjustment factors",
                   toolbar='below')
    hv.save(raw_and_af, 'localization_adjfactors_{}.html'.format(stn), resources="INLINE")

In [ ]:
selections = ck.Select()

# Import stations names and coordinates file

stations_df = read_csv_file("/home/jovyan/cae-notebooks/collaborative/DFU/wecc-station-data.csv")
all_stns = [s for s in stations_df['station id'].str.replace('-', '')] # list of all hadisd stations

for stn in all_stns[:2]:
    
    #=====================================================================================
    # STEP 1: RETRIEVE STATION OBSERVATION DATA AND CONVERT TIMEZONE
    filepaths = ["s3://cadcat/hadisd/HadISD_{}.zarr".format(stn)]
    print('Running bias-correction for: {}'.format(stn))

    obs_ds = xr.open_mfdataset(
        filepaths, engine="zarr",
        consolidated=False, parallel=True,
        backend_kwargs=dict(storage_options={"anon": True}))

    obs_ds = obs_ds.tas
    obs_ds = convert_units_to(obs_ds, "degF")
    obs_ds = obs_ds.chunk(dict(time=-1)).compute()
    lat0 = obs_ds.latitude.values
    lon0 = obs_ds.longitude.values
    
    # station timezone conversion
    obs_ds_data = obs_ds.loc[(obs_ds.time.dt.year >= 1981) & (obs_ds.time.dt.year <= 2014)]
    obs_ds_data.time # to keep
    obs_ds_tzslice = obs_ds.loc[obs_ds.time.dt.year == 2015]
    obs_ds_total = xr.concat([obs_ds_data, obs_ds_tzslice], dim='time') # combine and convert
    tf = TimezoneFinder()
    local_tz = tf.timezone_at(lng=lon0, lat=lat0)
    new_time = (pd.DatetimeIndex(obs_ds_total.time)
                .tz_localize("UTC")
                .tz_convert(local_tz)
                .tz_localize(None)
                .astype("datetime64[ns]"))
    obs_ds_total['time'] = new_time
    start = obs_ds_data.time[0]
    end = obs_ds_data.time[-1]
    obs_ds = obs_ds_total.sel(time=slice(start, end))
    obs_ds = obs_ds.drop_duplicates(dim='time', keep='first')
    obs_ds = convert_calendar(obs_ds, "noleap")
    print('Station observations ready for localization')
    
    #=====================================================================================
    # STEP 2: RETRIEVE GRIDDED MODEL DATA AND CONVERT TIMEZONE
    selections.scenario_historical=['Historical Climate']
    selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
    selections.append_historical = True
    selections.area_average = 'No'
    selections.time_slice = (1981, 2060)
    selections.resolution = '9 km' # cannot be 3km for outside of CA, keep consistent 
    selections.timescale = 'hourly'
    selections.variable = 'Air Temperature at 2m'
    selections.units = 'degF'
    selections.area_subset = 'lat/lon'
    selections.cached_area = ['coordinate selection']
    selections.latitude= (lat0-.05, lat0+.05) # for 9km spacing
    selections.longitude= (lon0-.05, lon0+.05) # for 9km spacing
    wrf_ds = selections.retrieve()
    
    # convert to local timezone for WRF data
    wrf_ds = convert_to_local_time(wrf_ds, selections)
    wrf_ds = get_closest_gridcell(wrf_ds, lat0, lon0)
    wrf_ds = wrf_ds.chunk(dict(time=-1)).compute() # need to unchunk for bias correction
    wrf_ds.attrs['physical_variable'] = wrf_ds.name
    wrf_ds.name = 'Raw'

    # # Select just CESM2 simulation -- WILL NEED ALL MODELS
    # cesm_sim_name = [sim for sim in wrf_ds.simulation.values if "cesm2" in sim.lower()]
    # wrf_ds = wrf_ds.sel(simulation = cesm_sim_name).squeeze()
    
    # dropping duplicate time indexes and resetting calendar
    wrf_ds = wrf_ds.drop_duplicates(dim='time', keep='first')
    wrf_ds = convert_calendar(wrf_ds, "noleap")
    print('Model data ready for localization')
    
    #=====================================================================================
    # STEP 3: QDM
    adj_factors, adj_ds = do_QDM(obs_ds, wrf_ds)
    
    # produce figure to save
    figure_to_export(adj_factors, stn)
    
    #=====================================================================================
    # STEP 4: EXPORT DATA TO NETCDF FILE
    # drop raw data variable and unnecessary coordinates, and rename adjusted back to air temperature
    adj_ds = adj_ds['Adjusted'] 
    adj_ds.name = 'Adjusted Air Temperature at 2m'
    adj_ds = adj_ds.reset_coords(names=['Lambert_Conformal','x', 'y', 'lakemask', 'landmask', 'lat', 'lon'], drop='True')

    # export data
    filename = 'localized_airtemp_{}.nc'.format(stn)
    ck.export(adj_ds, filename, 'NetCDF')
    
    #=====================================================================================
    # STEP 5: CLOSE DATASET TO SAVE MEMORY
    wrf_ds.close()
    obs_ds.close()
    
    # estimated time per station for 8 models per station: approx 5 min
    # approx 6 hours to generate all 71 stations